# 落子序列 Transformer 訓練與推論

執行 `aicup.train_seq_emb` 與 `aicup.infer_seq_emb`。checkpoint 存到 Drive 的 `player-trained-models/seq-<ARM>/`，嵌入快取存到同目錄的 `emb-cache/`。

- T4 跑不動
- 依序執行全部 cell；訓練中斷後重跑同一個 `ARM` 會從 `last.pt` 接續，接續時 `COMMIT` 要和開始時相同
- 每個 `ARM` 先 `STAGE = 'smoke'`，再 `'train'`，最後 `'infer'`

競賽一最終版本用到全部 7 個 `ARM`，都用雙方的手做 pooling（`--pool-mode all`）、訓練 12 epoch：

| ARM | 內容 |
|---|---|
| `state` | 規則狀態 token |
| `state_conf` | `state` 加 Conformer 卷積 |
| `patch` | `state` 加 7×7 局部盤面 |
| `patch_conv` | `patch` 加 Conformer 卷積與空間 bias |
| `patch_crop` | `patch_conv` 加前綴截斷與黑白各半抽樣 |
| `patch_d4` | `patch_conv` 加 D4 增強與 8× TTA |
| `patch_d4_val648` | `patch_d4` 改用 seed 23、不用 fp16，訓練資料接上 val648 |

`patch_conv_wide`（`patch_conv` 加多段池化、512 維嵌入）是失敗的實驗，不在最終版本流程內。

- `patch_d4_val648` 的 `train` 在 `/content` 把 val648 接在 train 後面（`aicup.concat_split`），Drive 上的檔案不變；`infer` 讀原本的目錄。
- `infer` 依序產生 newdev、val648、封存組與 A、B 組題目的嵌入。

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下，在本機產生後上傳。兩個打包目錄都依賴 `rank-features-v180000/` 與玩家清單，產生方式見 README。

| 路徑 | ARM、STAGE | 產生方式（本機，repo 根目錄） |
|---|---|---|
| `move-seq-state-v180000/train.npz`、`newdev.npz`、`val648.npz`、`pat3_vocab.npy` | 全部 | `python -m aicup.pack_move_seq --state-feats --out-dir move-seq-state-v180000` |
| `move-seq-state-v180000/hold.npz` | `infer` | `python -m aicup.pack_move_seq --state-feats --extra-split hold=humansl-analysis/holdout-v180000.csv --splits hold --pat3-vocab move-seq-state-v180000/pat3_vocab.npy --out-dir move-seq-state-v180000` |
| `move-seq-patch-v180000/train_patch.npz`、`newdev_patch.npz`、`val648_patch.npz` | `patch*` | `python -m aicup.pack_move_patch --seq-dir move-seq-state-v180000 --out-dir move-seq-patch-v180000` |
| `move-seq-patch-v180000/hold_patch.npz` | `patch*` 的 `infer` | `python -m aicup.pack_move_patch --seq-dir move-seq-state-v180000 --out-dir move-seq-patch-v180000 --splits hold` |
| `tests/player_identification_*.csv`（8 個） | `infer` | AI CUP 競賽平台 |
| `player-trained-models/seq-<ARM>/best_model.pth` | `infer` | 同一個 `ARM` 的 `train` |

## 參數

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
ARM = 'state'
STAGE = 'smoke'  # 'smoke'、'train' 或 'infer'
EPOCHS = 12
AUTO_UNASSIGN = True

SEQ, PATCH = 'move-seq-state-v180000', 'move-seq-patch-v180000'
CONV = ['--conv-kernel', '9', '--spatial-bias']
D4 = ['--augment-d4', '--val-tta', '8']
ARMS = {  # ARM: (--fp16, 用局部盤面, 其他旗標)
    'state': (False, False, []),
    'state_conf': (False, False, ['--conv-kernel', '9']),
    'patch': (True, True, []),
    'patch_conv': (True, True, CONV),
    'patch_crop': (True, True, CONV + ['--prefix-crop-prob', '0.2', '--prefix-crop-range', '80', '150',
                                       '--color-balance']),
    'patch_d4': (True, True, CONV + D4),
    'patch_d4_val648': (False, True, CONV + D4 + ['--seed', '23']),
    'patch_conv_wide': (True, True, CONV + ['--stage-pool', '--embed-dim', '512']),
}
if ARM not in ARMS or STAGE not in ('smoke', 'train', 'infer'):
    raise SystemExit('ARM 要是 %s 之一，STAGE 要是 smoke、train 或 infer' % ', '.join(ARMS))
FP16, USE_PATCH, FLAGS = ARMS[ARM]
CONCAT = ARM == 'patch_d4_val648' and STAGE != 'infer'
TRAIN_SEQ, TRAIN_PATCH = ('/content/seq-val648', '/content/patch-val648') if CONCAT else (SEQ, PATCH)
CONCAT_GAMES = 918278  # train 接上 val648 後的局數
RUN_DIR = 'player-trained-models/seq-' + ARM + ('-smoke' if STAGE == 'smoke' else '')
CACHE = RUN_DIR + '/emb-cache'
CKPT = RUN_DIR + '/best_model.pth'

if STAGE in ('smoke', 'train'):
    CMDS = [['python', '-m', 'aicup.train_seq_emb', '--seq-dir', TRAIN_SEQ, '--ckpt-dir', RUN_DIR,
             '--epochs', '1' if STAGE == 'smoke' else str(EPOCHS), *(['--fp16'] if FP16 else []), '--state-feats',
             *(['--patch-dir', TRAIN_PATCH] if USE_PATCH else []), *FLAGS, '--pool-mode', 'all',
             *(['--max-steps', '200', '--log-every', '100'] if STAGE == 'smoke' else [])]]
else:
    TTA = ['--tta', '8'] if '--augment-d4' in FLAGS else []
    PD = (['--patch-dir', PATCH] if USE_PATCH else []) + TTA
    INFER = ['python', '-m', 'aicup.infer_seq_emb', '--ckpt', CKPT]
    CMDS = [INFER + ['--mode', 'packed', '--seq-dir', SEQ, '--split', split, *PD, '--out', '%s/%s_emb.npz' % (CACHE, split)]
            for split in ('newdev', 'val648')]
    CMDS.append(INFER + ['--mode', 'packed', '--seq-npz', SEQ + '/hold.npz', *PD, '--out', CACHE + '/hold_emb.npz'])
    CMDS.append(INFER + ['--mode', 'test', '--groups', 'all', '--tests-dir', 'tests', '--out-dir', CACHE, *TTA])

SEQ_HINT = 'python -m aicup.pack_move_seq --state-feats --out-dir ' + SEQ
PATCH_HINT = 'python -m aicup.pack_move_patch --seq-dir %s --out-dir %s' % (SEQ, PATCH)
NEEDS = [(SEQ + '/' + f, SEQ_HINT) for f in ('train.npz', 'newdev.npz', 'val648.npz', 'pat3_vocab.npy')]
if USE_PATCH:
    NEEDS += [(PATCH + '/%s_patch.npz' % s, PATCH_HINT) for s in ('train', 'newdev', 'val648')]
if STAGE == 'infer':
    NEEDS.append((SEQ + '/hold.npz', SEQ_HINT + ' --extra-split hold=humansl-analysis/holdout-v180000.csv '
                  '--splits hold --pat3-vocab %s/pat3_vocab.npy' % SEQ))
    if USE_PATCH:
        NEEDS.append((PATCH + '/hold_patch.npz', PATCH_HINT + ' --splits hold'))
    NEEDS += [('tests/player_identification_%s%s_%s.csv' % (kind, g, split), 'AI CUP 競賽平台')
              for g in ('', '2') for kind in ('test', 'candidates') for split in ('public', 'private')]
    NEEDS.append((CKPT, '這份 notebook 同一個 ARM 的 STAGE train'))
for cmd in CMDS:
    print(' '.join(cmd))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：Drive 目錄的連結、需要的檔案都在。`patch_d4_val648` 的 `smoke`、`train` 另外印出串接後的局數。

In [ ]:
link_drive([SEQ, PATCH, 'tests', 'player-trained-models'])
need(NEEDS)
if CONCAT and not (os.path.exists(TRAIN_SEQ + '/train.npz') and os.path.exists(TRAIN_PATCH + '/train_patch.npz')):
    run(['python', '-m', 'aicup.concat_split', '--seq-dir', SEQ, '--patch-dir', PATCH, '--extra', 'val648',
         '--out-seq', TRAIN_SEQ, '--out-patch', TRAIN_PATCH, '--expect', str(CONCAT_GAMES)])
if STAGE != 'infer':
    check_commit(RUN_DIR, GIT_HEAD)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 執行

輸出同時寫到 Drive 的 `<RUN_DIR>/train.log` 或 `infer.log`。

In [ ]:
for cmd in CMDS:
    run(cmd, os.path.join(RUN_DIR, 'infer.log' if STAGE == 'infer' else 'train.log'))

## 檢查

要看到：訓練印出 `history.csv` 的最後幾個 epoch；推論列出 `emb-cache/` 的 11 個檔案。

In [ ]:
import pandas as pd

if STAGE == 'infer':
    for name in sorted(os.listdir(CACHE)):
        print('%-26s %8.1f MB' % (name, os.path.getsize(os.path.join(CACHE, name)) / 1024 ** 2))
else:
    display(pd.read_csv(os.path.join(RUN_DIR, 'history.csv')).tail().round(4))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')